# 📘 Bài 1 — Claude API cơ bản

**Tuần 16**

> Chuẩn bị: `pip install -e ".[llm]"` rồi làm theo [SETUP.md mục 5](../../../SETUP.md) để lấy API key.
>
> 💡 **Notebook này chạy được cả khi chưa có key** — các ô gọi API sẽ tự bỏ qua và in thông báo.

In [ ]:
import os
import json
from pathlib import Path

# Nap .env neu co
try:
    from dotenv import load_dotenv
    for p in (Path(".env"), Path("../../../.env")):
        if p.exists():
            load_dotenv(p)
            break
except ImportError:
    pass

import anthropic

MODEL_RE = "claude-haiku-4-5"        # $1 vào / $5 ra  — mặc định của khoá học
MODEL_MANH = "claude-opus-5"         # $5 vào / $25 ra — chỉ dùng khi so sánh

CO_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic() if CO_KEY else None


def goi(**kwargs):
    """Gọi API nếu có key; nếu chưa có thì in thông báo và trả về None.

    Nhờ hàm này, notebook chạy được cả khi bạn chưa lấy API key —
    bạn vẫn đọc được code và giải thích.
    """
    if not CO_KEY:
        print("⏭  Bỏ qua — chưa có ANTHROPIC_API_KEY (xem SETUP.md mục 5)")
        return None
    return client.messages.create(**kwargs)


def van_ban(response) -> str:
    """Lấy toàn bộ text từ response. KHÔNG dùng response.content[0].text!"""
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


print("anthropic SDK:", anthropic.__version__)
print("API key      :", "đã có ✓" if CO_KEY else "CHƯA CÓ — notebook vẫn chạy được")

## 1. Cuộc gọi đầu tiên

In [ ]:
r = goi(
    model=MODEL_RE,
    max_tokens=300,
    messages=[{"role": "user", "content": "Giải thích RAG trong đúng 2 câu."}],
)

if r:
    print(van_ban(r))
    print()
    print(f"stop_reason : {r.stop_reason}")
    print(f"token vào   : {r.usage.input_tokens}")
    print(f"token ra    : {r.usage.output_tokens}")

## 2. ⚠️ `response.content` là một DANH SÁCH

In [ ]:
if r:
    print(f"Kiểu của content: {type(r.content).__name__}")
    print(f"Số khối: {len(r.content)}")
    for i, b in enumerate(r.content):
        print(f"  khối {i}: type={b.type!r}")

print("""
❌ SAI:  response.content[0].text
✅ ĐÚNG: "".join(b.text for b in response.content if b.type == "text")

Vì content có thể chứa: text, thinking, tool_use...
Nếu model trả về khối thinking trước, content[0].text nem AttributeError.
""")

> 🔑 Đây là lỗi phổ biến nhất của người mới dùng Claude API. Viết **một hàm tiện ích** như `van_ban()` ở trên và dùng nó ở mọi nơi — đừng bao giờ chạm vào `.content[0]`.

## 3. API là STATELESS — Claude không nhớ gì cả

In [ ]:
lich_su = []


def hoi(noi_dung: str, in_ra: bool = True) -> str:
    """Gửi TOÀN BỘ lịch sử mỗi lần — đó là ý nghĩa của 'stateless'."""
    lich_su.append({"role": "user", "content": noi_dung})

    r = goi(model=MODEL_RE, max_tokens=300, messages=lich_su)
    if r is None:
        lich_su.pop()
        return ""

    tra_loi = van_ban(r)
    lich_su.append({"role": "assistant", "content": tra_loi})

    if in_ra:
        print(f"👤 {noi_dung}")
        print(f"🤖 {tra_loi}")
        print(f"   ({r.usage.input_tokens} token vào)\n")
    return tra_loi


hoi("Tên tôi là Nam.")
hoi("Tên tôi là gì?")

In [ ]:
# Chứng minh: gọi MỚI hoàn toàn, không gửi lịch sử -> model không biết gì
r2 = goi(
    model=MODEL_RE,
    max_tokens=100,
    messages=[{"role": "user", "content": "Tên tôi là gì?"}],
)
if r2:
    print("Hỏi lại KHÔNG kèm lịch sử:")
    print(f"🤖 {van_ban(r2)}")
    print("\n-> Model KHÔNG nhớ. 'Trí nhớ' hoàn toàn do BẠN quản lý.")

**Ba hệ quả của stateless:**

| Hệ quả | Chi tiết |
|---|---|
| **Chi phí tăng theo bình phương** | Lượt N gửi lại N−1 lượt trước |
| **Có giới hạn** | Vượt context window → lỗi |
| **Bạn kiểm soát bộ nhớ** | Muốn model "quên" thì cắt bớt `messages` |

Hãy nhìn `token vào` tăng dần qua từng lượt ở ô trên — đó chính là chi phí đang tăng.

## 4. System prompt

In [ ]:
cau_hoi = "Làm sao đọc file JSON?"

for ten, sys_prompt in [
    ("KHÔNG có system", None),
    ("CÓ system", "Bạn là trợ lý kỹ thuật. Trả lời trong đúng 1 câu, kèm 1 dòng code Python."),
]:
    kwargs = {"model": MODEL_RE, "max_tokens": 300,
              "messages": [{"role": "user", "content": cau_hoi}]}
    if sys_prompt:
        kwargs["system"] = sys_prompt

    r = goi(**kwargs)
    if r:
        print(f"--- {ten} ({r.usage.output_tokens} token ra) ---")
        print(van_ban(r)[:400])
        print()

`system` là **tham số riêng**, không phải phần tử trong `messages`. Nó mang **thẩm quyền cao hơn** nội dung người dùng.

> 🔒 **Bảo mật:** đừng bao giờ nhét nội dung do người dùng cung cấp thẳng vào `system` — đó là cửa ngõ của **prompt injection**.

## 5. ⚠️ Tham số khác nhau giữa các thế hệ model

In [ ]:
print("""
temperature — độ ngẫu nhiên (bạn đã hiểu cơ chế ở Phase 06)
   ✅ Haiku 4.5, Sonnet 4.6, Opus 4.6
   ❌ Sonnet 5, Opus 5, Opus 4.8/4.7   -> LỖI 400

effort — độ sâu suy luận (low/medium/high/xhigh/max)
   ✅ Sonnet 5, Opus 5, Opus 4.8/4.7
   ❌ Haiku 4.5                         -> LỖI

=> Đổi model thì phải đổi cả tham số.
   Đây là lỗi rất hay gặp khi nâng cấp model.
""")

In [ ]:
# temperature TRÊN HAIKU — dùng được
for temp in [0.0, 1.0]:
    r = goi(
        model=MODEL_RE,
        max_tokens=60,
        temperature=temp,
        messages=[{"role": "user", "content": "Đặt tên cho một quán cà phê. Chỉ trả về tên."}],
    )
    if r:
        print(f"temperature={temp}: {van_ban(r).strip()}")

In [ ]:
# Chạy 3 lần với temperature=0 -> kết quả ổn định
if CO_KEY:
    for i in range(3):
        r = goi(model=MODEL_RE, max_tokens=60, temperature=0,
                messages=[{"role": "user", "content": "Đặt tên một quán cà phê. Chỉ trả về tên."}])
        print(f"  lần {i+1}: {van_ban(r).strip()}")
    print("\n-> temperature=0 cho kết quả TÁI LẬP ĐƯỢC.")
    print("   Đó là lý do luôn dùng temperature=0 khi làm EVAL (Tuần 19).")
else:
    print("⏭  Bỏ qua — chưa có API key")

## 6. ⭐ Đếm token & tính chi phí

In [ ]:
BANG_GIA = {
    "claude-haiku-4-5": {"vao": 1.00, "ra": 5.00},
    "claude-sonnet-5":  {"vao": 3.00, "ra": 15.00},
    "claude-opus-5":    {"vao": 5.00, "ra": 25.00},
}


def chi_phi(token_vao, token_ra, model):
    g = BANG_GIA[model]
    return token_vao / 1e6 * g["vao"] + token_ra / 1e6 * g["ra"]


print(f"  {'model':<20} {'1M vào + 1M ra':>16}")
print("  " + "-" * 38)
for m in BANG_GIA:
    print(f"  {m:<20} {'$' + format(chi_phi(1e6, 1e6, m), '.2f'):>16}")

print("\n  Token RA đắt gấp 5 lần token VÀO ở mọi model.")
print("  -> Rút ngắn CÂU TRẢ LỜI tiết kiệm hơn rút ngắn PROMPT.")

In [ ]:
# Đếm token TRƯỚC khi gọi — biết chi phí trước khi trả tiền
van_ban_dai = "Xin chào, tôi muốn hỏi về sản phẩm của bạn. " * 50

if CO_KEY:
    dem = client.messages.count_tokens(
        model=MODEL_RE,
        messages=[{"role": "user", "content": van_ban_dai}],
    )
    print(f"  Số ký tự  : {len(van_ban_dai):,}")
    print(f"  Số token  : {dem.input_tokens:,}")
    print(f"  Tỷ lệ     : {len(van_ban_dai)/dem.input_tokens:.2f} ký tự/token")
    print(f"  Chi phí vào: ${chi_phi(dem.input_tokens, 0, MODEL_RE):.6f}")
else:
    print("⏭  Bỏ qua — chưa có API key")

print("""
⚠️ KHÔNG dùng tiktoken để đếm token cho Claude.
   Đó là tokenizer của OpenAI và cho con số SAI.
   Dùng client.messages.count_tokens().
""")

In [ ]:
# Tiếng Việt vs tiếng Anh — cùng một ý
cap = [
    ("Xin chào, bạn khoẻ không? Hôm nay trời đẹp quá.",
     "Hello, how are you? The weather is beautiful today."),
    ("Tôi đang học lập trình Python để làm kỹ sư AI.",
     "I am learning Python programming to become an AI engineer."),
]

if CO_KEY:
    print(f"  {'':<52} {'token':>6}")
    for vi, en in cap:
        tv = client.messages.count_tokens(model=MODEL_RE,
                                          messages=[{"role": "user", "content": vi}]).input_tokens
        ta = client.messages.count_tokens(model=MODEL_RE,
                                          messages=[{"role": "user", "content": en}]).input_tokens
        print(f"  VI  {vi[:46]:<48} {tv:>6}")
        print(f"  EN  {en[:46]:<48} {ta:>6}")
        print(f"      -> tiếng Việt tốn nhiều hơn {tv/ta:.2f} lần\n")
else:
    print("⏭  Bỏ qua — chưa có API key")

> 🔗 Bạn đã học **vì sao** ở Phase 06: tokenizer được huấn luyện chủ yếu trên tiếng Anh, nên ký tự tiếng Việt có dấu bị tách vụn hơn.
>
> **Hệ quả thực tế:** cùng một ý, prompt tiếng Việt **tốn tiền hơn**. Một chiến lược thường dùng: viết *system prompt* bằng tiếng Anh (nó ổn định và lặp lại nhiều lần), để nội dung người dùng bằng tiếng Việt.

## 7. Chi phí hội thoại tăng theo BÌNH PHƯƠNG

In [ ]:
# Mô phỏng: mỗi lượt user ~50 token, assistant ~100 token
tong_vao, tong_chi_phi = 0, 0.0
print(f"  {'lượt':>5} {'token vào':>11} {'chi phí lượt':>14} {'tích luỹ':>12}")
print("  " + "-" * 46)

lich_su_token = 0
for luot in range(1, 11):
    lich_su_token += 50                      # tin nhắn user mới
    vao = lich_su_token
    ra = 100
    lich_su_token += ra                      # trả lời được cộng vào lịch sử

    cp = chi_phi(vao, ra, MODEL_RE)
    tong_chi_phi += cp
    tong_vao += vao
    if luot in (1, 2, 5, 10):
        print(f"  {luot:>5} {vao:>11,} {'$' + format(cp, '.6f'):>14} {'$' + format(tong_chi_phi, '.6f'):>12}")

print(f"\n  Lượt 10 tốn {chi_phi(lich_su_token-100, 100, MODEL_RE)/chi_phi(50, 100, MODEL_RE):.1f} lần lượt 1")
print("  -> Đó là lý do phải CẮT LỊCH SỬ (bài tập ex01).")

## 8. `stop_reason` — luôn kiểm tra

In [ ]:
# Cố ý đặt max_tokens quá nhỏ để thấy hiện tượng bị cắt
r_cut = goi(
    model=MODEL_RE,
    max_tokens=30,
    messages=[{"role": "user", "content": "Liệt kê 10 thành phố lớn của Việt Nam."}],
)

if r_cut:
    print(f"stop_reason: {r_cut.stop_reason}")
    print(f"Nội dung   : {van_ban(r_cut)!r}")
    print()
    if r_cut.stop_reason == "max_tokens":
        print("⚠️  BỊ CẮT GIỮA CHỪNG. Bạn đã trả tiền cho một kết quả hỏng.")
        print("    Rất nhiều bug 'JSON không parse được' thực ra là do đây.")

| `stop_reason` | Nghĩa | Hành động |
|---|---|---|
| `end_turn` | Xong bình thường | ✅ |
| `max_tokens` | **Bị cắt** | Tăng `max_tokens` |
| `stop_sequence` | Gặp chuỗi dừng | — |
| `tool_use` | Model muốn gọi tool | Thực thi rồi gọi tiếp |
| `refusal` | Từ chối vì an toàn | Đọc `stop_details` |

> 📌 Một dòng kiểm tra tiết kiệm cả buổi debug:
> ```python
> if r.stop_reason == "max_tokens":
>     raise ValueError("Đầu ra bị cắt — tăng max_tokens")
> ```

## 9. Xử lý lỗi

In [ ]:
# Cố ý dùng model ID SAI (có hậu tố ngày tháng — lỗi rất hay gặp)
if CO_KEY:
    try:
        client.messages.create(
            model="claude-haiku-4-5-20251001",     # ← SAI: thừa hậu tố ngày
            max_tokens=10,
            messages=[{"role": "user", "content": "hi"}],
        )
    except anthropic.NotFoundError as e:
        print(f"NotFoundError: {str(e)[:160]}")
    except anthropic.BadRequestError as e:
        print(f"BadRequestError: {str(e)[:160]}")
    print("\n⚠️  ID model KHÔNG có hậu tố ngày tháng. Đúng là: claude-haiku-4-5")
else:
    print("⏭  Bỏ qua — chưa có API key")
    print("⚠️  Nhớ: ID model KHÔNG có hậu tố ngày tháng.")

```python
try:
    r = client.messages.create(...)
except anthropic.BadRequestError as e:      # sai tham số
    ...
except anthropic.AuthenticationError:       # key sai
    ...
except anthropic.RateLimitError as e:       # 429 — CHỜ RỒI THỬ LẠI
    cho = int(e.response.headers.get("retry-after", "60"))
except anthropic.APIStatusError as e:       # 5xx — thử lại; 4xx khác — không
    ...
except anthropic.APIConnectionError:        # mạng
    ...
```

> ⚠️ **Bắt một chuỗi lỗi cụ thể, không bắt `Exception` chung.** Gộp tất cả làm mất phân biệt giữa lỗi **thử lại được** (429, 5xx, mạng) và lỗi **không thử lại được** (400, 404). Thử lại một request sai tham số chỉ tốn thời gian.

**SDK tự retry sẵn** 408/409/429/5xx với exponential backoff (mặc định 2 lần). Chỉnh bằng `anthropic.Anthropic(max_retries=5)`.

Khi báo lỗi cho Anthropic, gửi kèm `r._request_id`.

## 10. So sánh model — khi nào cần model mạnh?

In [ ]:
cau_hoi_kho = """Một cửa hàng bán 3 loại combo:
- Combo A: 2 bánh + 1 nước, giá 85k
- Combo B: 1 bánh + 2 nước, giá 75k
- Combo C: 3 bánh + 3 nước, giá 180k
Nếu mua lẻ: bánh 35k, nước 20k.
Combo nào tiết kiệm nhất so với mua lẻ, tính theo PHẦN TRĂM?"""

if CO_KEY:
    for ten, model in [("Haiku 4.5", MODEL_RE), ("Opus 5", MODEL_MANH)]:
        kwargs = {"model": model, "max_tokens": 600,
                  "messages": [{"role": "user", "content": cau_hoi_kho}]}
        # Opus 5 KHÔNG nhận temperature; Haiku 4.5 KHÔNG nhận effort
        if model == MODEL_RE:
            kwargs["temperature"] = 0

        r = goi(**kwargs)
        cp = chi_phi(r.usage.input_tokens, r.usage.output_tokens, model)
        print(f"--- {ten}  (${cp:.5f}) ---")
        print(van_ban(r)[-500:])
        print()
else:
    print("⏭  Bỏ qua — chưa có API key")

**Quy trình chọn model đúng đắn trong công việc thật:**

```
① Bắt đầu bằng model MẠNH NHẤT → biết trần chất lượng
② Xây bộ EVAL để đo (Tuần 19)
③ Thử hạ xuống model rẻ hơn, đo lại
④ Chỉ hạ nếu chất lượng vẫn đạt yêu cầu
```

> ⚠️ Chọn model rẻ **trước** rồi ngạc nhiên vì chất lượng là thứ tự sai — bạn sẽ không biết vấn đề nằm ở model hay ở prompt của mình.
>
> Khoá học này dùng Haiku mặc định vì bạn đang **học** và cần chạy hàng trăm thử nghiệm với ngân sách nhỏ. Đó là một quyết định có ý thức, không phải mặc định.

## ✍️ Bài tập

```bash
pytest tests/phase07/test_ex01.py -v      # không cần API key
```